In [1]:
pwd

'/data1/kishoretarafdar/src.port/NSLI.v00'

In [2]:
!python --version

Python 3.12.7


GPU availability?

In [3]:
import tensorflow as tf
print(f"TensorFlow version {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
len(gpus)

2025-06-06 01:42:07.550373: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749154327.573181 3080753 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749154327.580244 3080753 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-06 01:42:07.604417: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version 2.18.0
Num GPUs Available:  3


3

Select one GPU

        Restrict code to use a particular GPU...

In [4]:
# # include ../dirx 
mylibpath = [
    '/home/kishoretarafdar/bin',
    '/data1/kishoretarafdar/src.port/NSLI.v00/utils.Volterra'
    #'/home/k/PLAYGROUND10GB/SKULSTRIPpaper__'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath

from tf_select_a_gpu import select_a_gpu

In [5]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [6]:
# select_gpu = gpus[gpu_id]
memory_limit = 48#GB
select_a_gpu(gpus, gpu_id=2, memory_limit=memory_limit)
# del gpu_id, select_a_gpu, select_gpu

3 Physical GPUs available 
Selected 1 Logical GPU with 48 GB memory limit


I0000 00:00:1749154336.629758 3080753 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 49152 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


# Separable conv4d with 2d kernels

        Quadratic NLSI for 2D I/O

    
        Strategy tested with nonseparable conv2d 
        Perfect match with one channel input
        !! Does not match when multiple channel input

        !! Not possible to test the strategy with nonseparable high dimensional convolutions
        (apply update when a libray is located online for nonseparable 4d convolutions)

In [456]:
import tensorflow as tf
from tensorflow.keras.layers import Layer

class SeparableConv4D(Layer):
    """Separable 4D convolution using 2D kernels
    
    @kkt 06-06-2025"""
    def __init__(self, filters, kernel_size, **kwargs):
        super(SeparableConv4D, self).__init__(**kwargs)
        # self.filters = filters
        self.kernel_size = kernel_size
        
    def build(self, input_shape):
        self.filters = input_shape[-1]
        # Create a 2D kernel that will be applied to both spatial dimensions
        self.kernel = self.add_weight(
            name='kernel2d',
            shape=(self.kernel_size, self.kernel_size, input_shape[-1], self.filters),
            initializer='glorot_uniform',
            trainable=True
        )
        
    def call(self, inputs):
        return self.__separable_conv4d(inputs)
        

    # Function: 4D separable convolution using a single 2D kernel
    def __separable_conv4d(self, x):
        # x: shape [B, N1, N2, N3, N4, C]
        # B, N1, N2, N3, N4, C = x.shape
        # Get static shape for dimensions that shouldn't change
        input_shape = x.shape.as_list()
        N1, N2, N3, N4 = input_shape[1], input_shape[2], input_shape[3], input_shape[4]
        
        # Get dynamic batch size
        B = tf.shape(x)[0]

        ## Step 1: Convolve over (N1, N2)
        x1 = tf.reshape(x, [-1, N1, N2, self.filters])  # shape: (B*N3*N4, N1, N2, C)
        y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
        y1 = tf.reshape(y1, [B, N1, N2, N3, N4, self.filters])   # (B, N1, N2, N3, N4, C)
        y1 = tf.transpose(y1, perm=[0,3,4,1,2,5])
    
        
        # print('+y1 ', y1.shape)


        ## Step 2: Convolve over (N3, N4)
        x2 = tf.reshape(y1, [-1, N3, N4, self.filters])  # shape: (B*N1*N2, N3, N4, C)
        y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
        y2 = tf.reshape(y2, [B, N1, N2, N3, N4, self.filters])    # final shape
        y2 = tf.transpose(y2, perm=[0,3,4,1,2,5])

        return y2
        
    def get_kernel(self):
        """Returns the kernel weights as a numpy array"""
        return self.kernel.numpy()

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1], input_shape[2], input_shape[3], input_shape[4], self.filters)
    
    def get_config(self):
        config = super(SeparableConv4D, self).get_config()
        config.update({
            'filters': self.filters,
            'kernel_size': self.kernel_size
        })
        return config

# Create a model using the layer
inputs = tf.keras.Input(shape=(16, 16, 16, 16, 2))  # (N1, N2, N3, N4, C)
inputs = tf.keras.Input(shape=(128, 128, 128, 128, 2))  # (N1, N2, N3, N4, C)
outputs = SeparableConv4D(filters=6, kernel_size=3)(inputs)
model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.summary()

# # # Test with random data
# test_input = tf.random.normal([2, 16, 16, 16, 16, 2])
# output = model(test_input)
# print(output.shape)
# del outputs, model, inputs, test_input, output

Model: "functional_39"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_54 (InputLayer)     │ (None, 128, 128, 128,  │             0 │
│                                 │ 128, 2)                │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv4d_63             │ (None, 128, 128, 128,  │            36 │
│ (SeparableConv4D)               │ 128, 2)                │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36 (144.00 B)

 Trainable params: 36 (144.00 B)

 Non-trainable params: 0 (0.00 B)

In [448]:
# Input dimensions
B, N1, N2, N3, N4, C = 2, 8, 8, 8, 8, 3  # Batch size, 4D volume size, channels
# B, N1, N2, N3, N4, C = 1, 2, 2, 2, 2, 1  # Batch size, 4D volume size, channels
# B, N1, N2, N3, N4, C = 1, 3, 3, 2, 2, 1  # Batch size, 4D volume size, channels
x = tf.random.normal((B, N1, N2, N3, N4, C))
x.shape


TensorShape([2, 8, 8, 8, 8, 3])

In [449]:
layer = SeparableConv4D(filters=x.shape[-1], kernel_size=3)
yout = layer(x)
kernel2d = layer.get_kernel()
print("Kernel shape:", kernel2d.shape)  # Should be (3, 3, 2, 32)
yout.shape

Kernel shape: (3, 3, 3, 3)


TensorShape([2, 8, 8, 8, 8, 3])

In [450]:
# # Shared 2D convolution kernel (C -> C to keep channels same)
# kH, kW = 3, 3
# kH, kW = 2, 2
# kernel2d = tf.random.normal((kH, kW, C, C))  # single 2D kernel reused



# Run the separable 4D convolution
y = separable_conv4d(x, kernel2d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel2d.shape)
print("Output shape:", y.shape)


Input shape : (2, 8, 8, 8, 8, 3)
Kernel 2D shape: (3, 3, 3, 3)
Output shape: (2, 8, 8, 8, 8, 3)


In [451]:
# # Compare
diff = tf.reduce_max(tf.abs(y - yout))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(y - yout) < 1e-4).numpy())

Max absolute difference: 0.0
Outputs match: True


        CHECK PASS: both layer and function giving same outpput

In [ ]:
## 
break

### separable_conv4d main

In [446]:
import tensorflow as tf

# Function: 4D separable convolution using a single 2D kernel
def separable_conv4d(x, kernel2d):
    # x: shape [B, N1, N2, N3, N4, C]
    B, N1, N2, N3, N4, C = x.shape

    ## Step 1: Convolve over (N1, N2)
    x1 = tf.reshape(x, [B * N3 * N4, N1, N2, C])  # shape: (B*N3*N4, N1, N2, C)
    y1 = tf.nn.convolution(x1, kernel2d, padding='SAME')
    y1 = tf.reshape(y1, [B, N1, N2, N3, N4, C])   # (B, N1, N2, N3, N4, C)
    y1 = tf.transpose(y1, perm=[0,3,4,1,2,5])
   
    
    # print('+y1 ', y1.shape)


    ## Step 2: Convolve over (N3, N4)
    x2 = tf.reshape(y1, [B * N1 * N2, N3, N4, C])  # shape: (B*N1*N2, N3, N4, C)
    y2 = tf.nn.convolution(x2, kernel2d, padding='SAME')
    y2 = tf.reshape(y2, [B, N1, N2, N3, N4, C])    # final shape
    y2 = tf.transpose(y2, perm=[0,3,4,1,2,5])

    return y2

# -------------------------------
# Test the function with dummy data
# -------------------------------

# Input dimensions
B, N1, N2, N3, N4, C = 2, 8, 8, 8, 8, 3  # Batch size, 4D volume size, channels
B, N1, N2, N3, N4, C = 1, 2, 2, 2, 2, 1  # Batch size, 4D volume size, channels
# B, N1, N2, N3, N4, C = 1, 3, 3, 2, 2, 1  # Batch size, 4D volume size, channels
x = tf.random.normal((B, N1, N2, N3, N4, C))

# Shared 2D convolution kernel (C -> C to keep channels same)
kH, kW = 3, 3
kH, kW = 2, 2
kernel2d = tf.random.normal((kH, kW, C, C))  # single 2D kernel reused

# Run the separable 4D convolution
y = separable_conv4d(x, kernel2d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel2d.shape)
print("Output shape:", y.shape)



Input shape : (1, 2, 2, 2, 2, 1)
Kernel 2D shape: (2, 2, 1, 1)
Output shape: (1, 2, 2, 2, 2, 1)


In [408]:
B, N1, N2, N3, N4, C = x.shape

_ = tf.reshape(x, [B * N3 * N4, N1, N2, C])  # shape: (B*N3*N4, N1, N2, C)
# print(f'x {x.shape} \nx1 ', x1.shape)
# y1 = tf.nn.convolution(x1, kernel2d, padding='SAME')
# print('y1 ', y1.shape)
_r = tf.reshape(_, [B, N1, N2, N3, N4, C]) 
x.shape, _r.shape


(TensorShape([1, 2, 2, 2, 2, 1]), TensorShape([1, 2, 2, 2, 2, 1]))

In [409]:

# # Compare
diff = tf.reduce_max(tf.abs(x - _r))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(x - _r) < 1e-4).numpy())


Max absolute difference: 0.0
Outputs match: True


In [410]:
## Step 2: Convolve over (N3, N4)
__ = tf.reshape(x, [B * N1 * N2, N3, N4, C])  # shape: (B*N1*N2, N3, N4, C)
# y2 = tf.nn.convolution(x2, kernel2d, padding='SAME')
__r = tf.reshape(__, [B, N1, N2, N3, N4, C])    # final shape
x.shape, __r.shape

(TensorShape([1, 2, 2, 2, 2, 1]), TensorShape([1, 2, 2, 2, 2, 1]))

In [411]:
# # Compare
diff = tf.reduce_max(tf.abs(x - __r))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(x - __r) < 1e-4).numpy())

Max absolute difference: 0.0
Outputs match: True


In [412]:
# print(f'x {x.shape} \nx1 ', x1.shape)
# y1 = tf.nn.convolution(x1, kernel2d, padding='SAME')
# print('y1 ', y1.shape)
_r = tf.reshape(_, [B, N3, N4, N1, N2, C]) 

    alternate method check

In [273]:
kernel4d = np.einsum('ijco,klco->ijklco', kernel2d, kernel2d)
kernel4d.shape

(2, 2, 2, 2, 1, 1)

In [ ]:
# Compute brute-force
# y_ref = tf.einsum('bijklc,ijklco->bijklo', x, kernel4d)



In [276]:
# Compute fast method
# y_fast = separable_4d_conv(x, kernel2d)

# # Compare
# diff = tf.reduce_max(tf.abs(y_ref - y_fast))
# print("Max absolute difference:", diff.numpy())
# print("Outputs match:", tf.reduce_all(tf.abs(y_ref - y_fast) < 1e-4).numpy())


In [ ]:
# np.squeeze(y_ref)

In [ ]:
# y_fast

In [ ]:
# tf.einsum('bijklc,ijklco->bijklo', x, kernel4d)

In [ ]:
# for b in range(x.shape[0]):
#   for n1 in range(x.shape[1]):
#     for n2 in range(x.shape[2]):
#       for n3 in range(x.shape[3]):
#         for n4 in range(x.shape[4]):
#           y[b,n1,n2,n3,n4,c], 

    conv 6d with 2d separable kernel

In [ ]:
import tensorflow as tf

def separable_6d_conv(x, kernel2d):
    # x: [B, N1, N2, N3, N4, N5, N6, C]
    B, N1, N2, N3, N4, N5, N6, C = x.shape
    O = kernel2d.shape[-1]

    # Step 1: Convolve over (N1, N2)
    x1 = tf.reshape(x, [B * N3 * N4 * N5 * N6, N1, N2, C])
    y1 = tf.nn.convolution(x1, kernel2d, padding='SAME')
    print(y1.shape)
    y1 = tf.reshape(y1, [B, N1, N2, N3, N4, N5, N6, C])
    # y1 = tf.reshape(y1, [B, N3, N4, N5, N6, N1, N2, C])
    # y1 = tf.transpose(y1, [0, 5, 6, 1, 2, 3, 4, 7])  # [B, N1, N2, N3, N4, N5, N6, C]
    # y1 = tf.transpose(y1, perm=[0,4,5,6,1,2,3,7])
    # y1 = tf.transpose(y1, perm=[0,1,2,3,4,5,6,7])

    # Step 2: Convolve over (N3, N4)
    x2 = tf.reshape(y1, [B * N1 * N2 * N5 * N6, N3, N4, C])
    y2 = tf.nn.convolution(x2, kernel2d, padding='SAME')
    y2 = tf.reshape(y2, [B, N1, N2, N3, N4, N5, N6, C])
    # y2 = tf.reshape(y2, [B, N1, N2, N5, N6, N3, N4, C])
    y2 = tf.transpose(y2, [0, 1, 2, 5, 6, 3, 4, 7])  # [B, N1, N2, N3, N4, N5, N6, C]

    # Step 3: Convolve over (N5, N6)
    x3 = tf.reshape(y2, [B * N1 * N2 * N3 * N4, N5, N6, C])
    y3 = tf.nn.convolution(x3, kernel2d, padding='SAME')
    y3 = tf.reshape(y3, [B, N1, N2, N3, N4, N5, N6, C])

    return y3

# Input shape
B, N1, N2, N3, N4, N5, N6, C = 1, 2, 2, 2, 2, 2, 2, 1
x = tf.random.normal((B, N1, N2, N3, N4, N5, N6, C))
# O = 2

# Shared kernel
kH, kW = 2, 2
kernel2d = tf.random.normal((kH, kW, C, C))

# Apply 6D separable conv
y = separable_6d_conv(x, kernel2d)

# Output
print("Input shape :", x.shape)
print("Kernel shape:", kernel2d.shape)
print("Output shape:", y.shape)


(16, 2, 2, 2)
Input shape : (1, 2, 2, 2, 2, 2, 2, 1)
Kernel shape: (2, 2, 1, 2)
Output shape: (1, 2, 2, 2, 2, 2, 2, 2)


# Separable 6D convolution with 3D kernel

    Quadratic NLSI for 3D I/O

In [462]:
import tensorflow as tf
from tensorflow.keras.layers import Layer

class SeparableConv6D(Layer):
    """Separable 6D convolution using 3D kernels
    
    @kkt 06-06-2025"""
    def __init__(self, filters, kernel_size, **kwargs):
        super(SeparableConv6D, self).__init__(**kwargs)
        # self.filters = filters
        self.kernel_size = kernel_size
        
    def build(self, input_shape):
        self.inchannels = input_shape[-1] 
        self.filters = input_shape[-1]
        # Create a 3D kernel that will be applied to both spatial dimensions
        self.kernel = self.add_weight(
            name='kernel3d',
            shape=(self.kernel_size, self.kernel_size, self.kernel_size, input_shape[-1], self.filters),
            initializer='glorot_uniform',
            trainable=True
        )
        
    def call(self, inputs):
        return self.__separable_conv6d(inputs)


    def __separable_conv6d(self, x):
        # x: [B, N1, N2, N3, N4, N5, N6, C]
        input_shape = x.shape.as_list()
        N1, N2, N3, N4, N5, N6 = input_shape[1], input_shape[2], input_shape[3], input_shape[4], input_shape[5], input_shape[6]
        B = tf.shape(x)[0]

        # O = kernel3d.shape[-1]
        # h2 = kernel3d

        # Step 1: Convolve over (N1, N2, N3)
        x1 = tf.reshape(x, [B * N4 * N5 * N6, N1, N2, N3, self.inchannels])
        # print('x', x.shape)
        # print(x1.shape)
        y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
        # print(y1.shape)
        y1 = tf.reshape(y1, [B, N1, N2, N3, N4, N5, N6, self.inchannels])
        # print(y1.shape)
        y1 = tf.transpose(y1, perm=[0,4,5,6,1,2,3,7])
    
        # Step 2: Convolve over (N4, N5, N6)
        x2 = tf.reshape(y1, [B * N1 * N2 * N3, N4, N5, N6, self.inchannels])
        # print(x2.shape)
        y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
        # print(y2.shape)
        y2 = tf.reshape(y2, [B, N1, N2, N3, N4, N5, N6, self.inchannels])
        # print(y2.shape)
        y2 = tf.transpose(y2, perm=[0,4,5,6,1,2,3,7])

        return y2
        

    # # Function: 4D separable convolution using a single 2D kernel
    # def __separable_conv4d(self, x):
    #     # x: shape [B, N1, N2, N3, N4, C]
    #     # B, N1, N2, N3, N4, C = x.shape
    #     # Get static shape for dimensions that shouldn't change
    #     input_shape = x.shape.as_list()
    #     N1, N2, N3, N4, N3,  = input_shape[1], input_shape[2], input_shape[3], input_shape[4]
        
    #     # Get dynamic batch size
    #     B = tf.shape(x)[0]

    #     ## Step 1: Convolve over (N1, N2)
    #     x1 = tf.reshape(x, [-1, N1, N2, self.filters])  # shape: (B*N3*N4, N1, N2, C)
    #     y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
    #     y1 = tf.reshape(y1, [B, N1, N2, N3, N4, self.filters])   # (B, N1, N2, N3, N4, C)
    #     y1 = tf.transpose(y1, perm=[0,3,4,1,2,5])
    
        
    #     # print('+y1 ', y1.shape)


    #     ## Step 2: Convolve over (N3, N4)
    #     x2 = tf.reshape(y1, [-1, N3, N4, self.filters])  # shape: (B*N1*N2, N3, N4, C)
    #     y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
    #     y2 = tf.reshape(y2, [B, N1, N2, N3, N4, self.filters])    # final shape
    #     y2 = tf.transpose(y2, perm=[0,3,4,1,2,5])

    #     return y2
        
    def get_kernel(self):
        """Returns the kernel weights as a numpy array"""
        return self.kernel.numpy()

    # def compute_output_shape(self, input_shape):
    #     return (input_shape[0], input_shape[1], input_shape[2], input_shape[3], input_shape[4], self.filters)
    
    def get_config(self):
        config = super(SeparableConv6D, self).get_config()
        config.update({
            'filters': self.filters,
            'kernel_size': self.kernel_size
        })
        return config

# Create a model using the layer
inputs = tf.keras.Input(shape=(16, 16, 16, 16, 16, 16, 2))  # (N1, N2, N3, N4, C)
# inputs = tf.keras.Input(shape=(16, 16, 16, 16, 16, 16, 2))  # (N1, N2, N3, N4, C)
outputs = SeparableConv6D(filters=6, kernel_size=3)(inputs)
model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.summary()

# # Test with random data
test_input = tf.random.normal([2, 16, 16, 16, 16, 16, 16, 2])
output = model(test_input)
print(output.shape)
del outputs, model, inputs, test_input, output

Model: "functional_41"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_60 (InputLayer)     │ (None, 16, 16, 16, 16, │             0 │
│                                 │ 16, 16, 2)             │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv6d_4              │ (None, 16, 16, 16, 16, │           108 │
│ (SeparableConv6D)               │ 16, 16, 2)             │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 108 (432.00 B)

 Trainable params: 108 (432.00 B)

 Non-trainable params: 0 (0.00 B)

(2, 16, 16, 16, 16, 16, 16, 2)


In [464]:
# Input dimensions
B, N1, N2, N3, N4, N5, N6, C = 2, 8, 8, 8, 8, 8, 8, 3  # Batch size, 4D volume size, channels
# B, N1, N2, N3, N4, C = 1, 2, 2, 2, 2, 1  # Batch size, 4D volume size, channels
# B, N1, N2, N3, N4, C = 1, 3, 3, 2, 2, 1  # Batch size, 4D volume size, channels
x = tf.random.normal((B, N1, N2, N3, N4, N5, N6, C))
x.shape


TensorShape([2, 8, 8, 8, 8, 8, 8, 3])

In [465]:
layer = SeparableConv6D(filters=x.shape[-1], kernel_size=3)
yout = layer(x)
kernel3d = layer.get_kernel()
print("Kernel shape:", kernel3d.shape)  # Should be (3, 3, 2, 32)
yout.shape

Kernel shape: (3, 3, 3, 3, 3)


TensorShape([2, 8, 8, 8, 8, 8, 8, 3])

In [466]:
# # Shared 2D convolution kernel (C -> C to keep channels same)
# kH, kW = 3, 3
# kH, kW = 2, 2
# kernel2d = tf.random.normal((kH, kW, C, C))  # single 2D kernel reused



# Run the separable 4D convolution
y = separable_conv6d(x, kernel3d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel3d.shape)
print("Output shape:", y.shape)


x (2, 8, 8, 8, 8, 8, 8, 3)
(1024, 8, 8, 8, 3)
(1024, 8, 8, 8, 3)
(2, 8, 8, 8, 8, 8, 8, 3)
(1024, 8, 8, 8, 3)
(1024, 8, 8, 8, 3)
(2, 8, 8, 8, 8, 8, 8, 3)
Input shape : (2, 8, 8, 8, 8, 8, 8, 3)
Kernel 2D shape: (3, 3, 3, 3, 3)
Output shape: (2, 8, 8, 8, 8, 8, 8, 3)


In [467]:
# # Compare
diff = tf.reduce_max(tf.abs(y - yout))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(y - yout) < 1e-4).numpy())

Max absolute difference: 0.0
Outputs match: True


### separable_conv6d main

In [463]:
import tensorflow as tf

def separable_conv6d(x, kernel3d):
    # x: [B, N1, N2, N3, N4, N5, N6, C]
    B, N1, N2, N3, N4, N5, N6, C = x.shape
    # O = kernel3d.shape[-1]
    h2 = kernel3d

    # Step 1: Convolve over (N1, N2, N3)
    x1 = tf.reshape(x, [B * N4 * N5 * N6, N1, N2, N3, C])
    print('x', x.shape)
    print(x1.shape)
    y1 = tf.nn.convolution(x1, h2, padding='SAME')
    print(y1.shape)
    y1 = tf.reshape(y1, [B, N1, N2, N3, N4, N5, N6, C])
    print(y1.shape)
    y1 = tf.transpose(y1, perm=[0,4,5,6,1,2,3,7])
   
    # Step 2: Convolve over (N4, N5, N6)
    x2 = tf.reshape(y1, [B * N1 * N2 * N3, N4, N5, N6, C])
    print(x2.shape)
    y2 = tf.nn.convolution(x2, h2, padding='SAME')
    print(y2.shape)
    y2 = tf.reshape(y2, [B, N1, N2, N3, N4, N5, N6, C])
    print(y2.shape)
    y2 = tf.transpose(y2, perm=[0,4,5,6,1,2,3,7])

    return y2

# -------------------------------
# Test the function with dummy data
# -------------------------------

# Input shape
B, N1, N2, N3, N4, N5, N6, C = 1, 2, 2, 2, 2, 2, 2, 1
# B, N1, N2, N3, N4, N5, N6, C = 2, 2, 2, 2, 2, 2, 2, 2
B, N1, N2, N3, N4, N5, N6, C = 5, 2, 3, 4, 5, 6, 7, 8
# B, N1, N2, N3, N4, N5, N6, C = 5, 8, 8, 8, 8, 8, 8, 2

# O = 4
x = tf.random.normal((B, N1, N2, N3, N4, N5, N6, C))

# Shared 2D convolution kernel (C -> C to keep channels same)
kH, kW, kD = 3, 3, 3
kH, kW, kD = 2, 2, 2
kernel3d = tf.random.normal((kH, kW, kD, C, C))  # single 2D kernel reused

# Run the separable 4D convolution
y = separable_conv6d(x, kernel3d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel3d.shape)
print("Output shape:", y.shape)



x (5, 2, 3, 4, 5, 6, 7, 8)
(1050, 2, 3, 4, 8)
(1050, 2, 3, 4, 8)
(5, 2, 3, 4, 5, 6, 7, 8)
(120, 5, 6, 7, 8)
(120, 5, 6, 7, 8)
(5, 2, 3, 4, 5, 6, 7, 8)
Input shape : (5, 2, 3, 4, 5, 6, 7, 8)
Kernel 2D shape: (2, 2, 2, 8, 8)
Output shape: (5, 5, 6, 7, 2, 3, 4, 8)


In [71]:
# Input shape
B, N1, N2, N3, N4, N5, N6, C = 1, 2, 2, 2, 2, 2, 2, 1
B, N1, N2, N3, N4, N5, N6, C = 2, 2, 2, 2, 2, 2, 2, 2
B, N1, N2, N3, N4, N5, N6, C = 5, 2, 3, 4, 5, 6, 7, 8
# B, N1, N2, N3, N4, N5, N6, C = 5, 8, 8, 8, 8, 8, 8, 1

O = 16
x = tf.random.normal((B, N1, N2, N3, N4, N5, N6, C))

# Shared 2D convolution kernel (C -> C to keep channels same)
kH, kW, kD = 3, 3, 3
kH, kW, kD = 2, 2, 2
kernel3d = tf.random.normal((kH, kW, kD, C, O))  # single 2D kernel reused



# x: [B, N1, N2, N3, N4, N5, N6, C]
B, N1, N2, N3, N4, N5, N6, C = x.shape
O = kernel3d.shape[-1]
h2 = kernel3d

# Step 1: Convolve over (N1, N2, N3)
x1 = tf.reshape(x, [B * N4 * N5 * N6, N1, N2, N3, C])
print(x.shape)
print(x1.shape)
y1 = tf.nn.convolution(x1, h2, padding='SAME')
print(y1.shape)
y1 = tf.reshape(y1, [B, N1, N2, N3, N4, N5, N6, O])
print(y1.shape)

# Step 2: Convolve over (N4, N5, N6)
x2 = tf.reshape(y1, [B * N1 * N2 * N3, N4, N5, N6, O])
print('x2    ', x2.shape)
print('filter', h2.shape)
y2 = tf.nn.convolution(x2, h2, padding='SAME')
print(y2.shape)
y2 = tf.reshape(y2, [B, N1, N2, N3, N4, N5, N6, O])
print(y2.shape)

(5, 2, 3, 4, 5, 6, 7, 8)
(1050, 2, 3, 4, 8)
(1050, 2, 3, 4, 16)
(5, 2, 3, 4, 5, 6, 7, 16)
x2     (120, 5, 6, 7, 16)
filter (2, 2, 2, 8, 16)
(120, 5, 6, 7, 16)
(5, 2, 3, 4, 5, 6, 7, 16)


TensorShape([2, 2, 2, 1, 4])

# Separable convolution strategy
        
        separable and nonseparable (referencd) 2d convolution
        outer product of kernel
        Perfect match with one channel input
        !! does not match when multiple channel input

In [ ]:
import tensorflow as tf

def separable_2d_conv(x, kernel1d):
    # x: [B, N1, N2, N3, N4, N5, N6, C]
    B, N1, N2, C = x.shape

    # Step 1: Convolve over (N1)
    ## x: B, N1, N2, C 
    x1 = tf.reshape(x, [B * N2, N1, C])
    y1 = tf.nn.convolution(x1, kernel1d, padding='SAME')
    y1 = tf.reshape(y1, [B, N1, N2, C])
    y1 = tf.transpose(y1, perm=[0,2,1,3])
    ## B, N2, N1, C 
   
    # Step 2: Convolve over (N2)
    ## B, N2, N1, C
    x2 = tf.reshape(y1, [B * N1, N2, C])
    y2 = tf.nn.convolution(x2, kernel1d, padding='SAME')
    y2 = tf.reshape(y2, [B, N1, N2, C])
    y2 = tf.transpose(y2, perm=[0,2,1,3])
    ## B, N1, N2, C

    return y2

# -------------------------------
# Test the function with dummy data
# -------------------------------

# Input shape
B, N1, N2, C = 1, 8, 8, 1 #OK
# B, N1, N2, C = 1, 2, 2, 2 # 
# B, N1, N2, C = 50, 8, 8, 1 #OK
# B, N1, N2, C = 2, 2, 2, 2 #issue with multiple channels
# B, N1, N2, C = 5, 2, 3, 2
# B, N1, N2, C = 5, 8, 8, 2

# O = 2
x = tf.random.normal((B, N1, N2, C))

# Shared 2D convolution kernel (C -> C to keep channels same)
kH = 3
kH = 2
kernel1d = tf.random.normal((kH, C, C))  # single 2D kernel reused

# Run the separable 4D convolution
y = separable_2d_conv(x, kernel1d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel1d.shape)
print("Output shape:", y.shape)



Input shape : (1, 8, 8, 1)
Kernel 2D shape: (2, 1, 1)
Output shape: (1, 8, 8, 1)


In [ ]:
# def foo(x):
#     C = 1
#     tf.expand_dims(tf.unstack(x, axis=-1))

[<tf.Tensor: shape=(1, 2, 2), dtype=float32, numpy=
 array([[[ 0.96406895,  1.7949798 ],
         [-0.18434155, -0.16262998]]], dtype=float32)>,
 <tf.Tensor: shape=(1, 2, 2), dtype=float32, numpy=
 array([[[ 0.25728133, -0.1631882 ],
         [ 0.96027887,  0.08398907]]], dtype=float32)>]

In [318]:
kernel2d = tf.einsum('ico,kco->ikco', kernel1d, kernel1d)
kernel2d.shape

TensorShape([2, 2, 1, 1])

In [319]:
ydef = tf.nn.convolution(x, kernel2d, padding='SAME')
ydef.shape

TensorShape([1, 8, 8, 1])

In [320]:
# Compare
diff = tf.reduce_max(tf.abs(y - ydef))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(y - ydef) < 1e-4).numpy())


Max absolute difference: 4.7683716e-07
Outputs match: True


np problem with multiple batches

!!issue with more than 1 channels

In [196]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [203]:
import numpy as np
np.squeeze(kernel2d)#[::-1]

array([[[[ 0.8992288 ,  0.03782616],
         [ 0.02944931,  0.5439975 ]],

        [[-0.38017905,  0.05966839],
         [ 0.12532397,  0.58402634]]],


       [[[-0.38017905,  0.05966839],
         [ 0.12532397,  0.58402634]],

        [[ 0.1607334 ,  0.09412312],
         [ 0.5333265 ,  0.6270005 ]]]], dtype=float32)

In [119]:
np.squeeze(x)

array([[-1.3135974 ,  1.6594728 ],
       [-0.42444474, -0.9835947 ]], dtype=float32)

In [126]:
np.squeeze(kernel1d)

array([-2.7352364 ,  0.54115134], dtype=float32)

In [ ]:
np.squeeze(y)

array([[-14.74033  ,  12.415377 ],
       [ -0.2637028,  -7.358782 ]], dtype=float32)

    verifying ydef

In [121]:
np.squeeze(ydef)

array([[-11.943804,  13.871271],
       [ -1.719597,  -7.358782]], dtype=float32)

In [122]:
tf.einsum('ij,ij->', np.squeeze(x), np.squeeze(kernel2d)).numpy()

np.float32(-11.943804)

In [123]:
7.4815183*1.6594728 +-1.4801768*-0.9835947

13.8712701770952

In [124]:
7.4815183*-0.42444474 + -1.4801768*-0.9835947

-1.7195970341057818

In [125]:
-0.9835947 * 7.4815183

-7.35878174783301

    verified above that separable and nonseparable convolution gives same results

np.float32(-11.943804)

In [127]:
import numpy as np
np.squeeze(kernel2d)#[::-1]

array([[ 7.4815183 , -1.4801768 ],
       [-1.4801768 ,  0.29284477]], dtype=float32)

In [128]:
np.squeeze(x)

array([[-1.3135974 ,  1.6594728 ],
       [-0.42444474, -0.9835947 ]], dtype=float32)

In [129]:
np.squeeze(kernel1d)

array([-2.7352364 ,  0.54115134], dtype=float32)

In [130]:
x1 = tf.reshape(x, [B * N2, N1, C])
x1.shape

TensorShape([2, 2, 1])

In [145]:
y1 = tf.nn.convolution(x1, kernel1d, padding='SAME')
y1.shape
y1 = tf.reshape(y1, [B, N1, N2, C])
tf.squeeze(y1)

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[ 4.4910254 , -4.5390506 ],
       [ 0.62868315,  2.6903641 ]], dtype=float32)>

        CHECK

In [146]:
-2.7352364*-1.3135974 + 0.54115134*1.6594728

4.4910253528389115

In [147]:
-2.7352364*1.6594728

-4.53905040736992

In [148]:
-2.7352364*-0.42444474 + 0.54115134*-0.9835947

0.628683112714638

In [149]:
-2.7352364*-0.9835947

2.69036402628708

In [155]:
y1 = tf.transpose(y1, perm=[0,2,1,3])
x2 = tf.reshape(y1, [B * N1, N2, C])
tf.squeeze(x2)

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[ 4.4910254 ,  0.62868315],
       [-4.5390506 ,  2.6903641 ]], dtype=float32)>

In [156]:
x2[0,...]

<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[4.4910254 ],
       [0.62868315]], dtype=float32)>

In [159]:
y2 = tf.nn.convolution(x2, kernel1d, padding='SAME')
y2 = tf.reshape(y2, [B, N1, N2, C])
y2 = tf.transpose(y2, perm=[0,2,1,3])
tf.squeeze(y2)

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[-11.943804,  13.871271],
       [ -1.719597,  -7.358782]], dtype=float32)>

In [154]:
-2.7352364*4.4910254 +  0.54115134*0.62868315

-11.943803418346638

        issue with channels!!

In [228]:
kernel2d[:,:,0,0], kernel2d[:,:,1,0],

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 0.04413895, -0.24178982],
        [-0.24178982,  1.3245063 ]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[0.09562411, 0.0405251 ],
        [0.0405251 , 0.01717436]], dtype=float32)>)

In [229]:
kernel2d[:,:,0,1], kernel2d[:,:,1,1]

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 0.36423975, -0.15391304],
        [-0.15391304,  0.06503744]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[0.8157267 , 0.63113356],
        [0.63113356, 0.48831257]], dtype=float32)>)

In [ ]:
x[0,:,:,0], x[0,:,:,1], 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[-0.50684583, -1.3502959 ],
        [ 0.33504048,  0.5373964 ]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.1334256 , -0.25891632],
        [ 0.12293333,  0.20609343]], dtype=float32)>)

In [251]:
kernel1d[:,0,0], kernel1d[:,1,0]

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([ 0.21009271, -1.150872  ], dtype=float32)>,
 <tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.3092315, -0.131051 ], dtype=float32)>)

In [252]:
kernel1d[:,0,1], kernel1d[:,1,1]

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.6035228,  0.2550244], dtype=float32)>,
 <tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.9031759 , -0.69879365], dtype=float32)>)

In [232]:
y[0,:,:,0], y[0,:,:,1], 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.2576822 , -0.35678944],
        [-0.0297929 ,  0.16818404]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 0.1807254 , -0.45509022],
        [ 0.65922415,  0.4313671 ]], dtype=float32)>)

In [233]:
ydef[0,:,:,0], ydef[0,:,:,1], 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.0413033 , -0.2059443 ],
        [-0.09504129,  0.04342761]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 0.9459787 , -0.65567625],
        [ 0.26967523,  0.36385703]], dtype=float32)>)

In [244]:
##
# 0.04413895*-0.50684583 + -0.24178982*-1.3502959 + -0.24178982*0.33504048 + 1.3245063*0.5373964 + 
# 0.09562411*1.1334256 + 0.0405251*-0.25891632 + 0.0405251*0.12293333 + 0.01717436*0.20609343  

In [257]:
tf.einsum('ij,ij->', np.squeeze(kernel2d[:,:,0,0]), np.squeeze(x[0,:,:,0])).numpy() + tf.einsum('ij,ij->', np.squeeze(kernel2d[:,:,1,0]), np.squeeze(x[0,:,:,1])).numpy()

np.float32(1.0413033)

---

In [248]:
x1 = tf.reshape(x, [B * N2, N1, C])
y1 = tf.nn.convolution(x1, kernel1d, padding='SAME')
y1 = tf.reshape(y1, [B, N1, N2, C])
y1.shape

TensorShape([1, 2, 2, 2])

In [249]:
y1[0,:,:,0], y1[0,:,:,1] 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.1309735 , -0.20362225],
        [-0.61310846,  0.04917248]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[-0.881219  ,  1.0487813 ],
        [-0.32020256, -0.51046956]], dtype=float32)>)

In [253]:
kernel1d[:,0,0], kernel1d[:,1,0]

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([ 0.21009271, -1.150872  ], dtype=float32)>,
 <tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.3092315, -0.131051 ], dtype=float32)>)

In [254]:
kernel1d[:,0,1], kernel1d[:,1,1]

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.6035228,  0.2550244], dtype=float32)>,
 <tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.9031759 , -0.69879365], dtype=float32)>)

In [255]:
x[0,:,:,0], x[0,:,:,1], 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[-0.50684583, -1.3502959 ],
        [ 0.33504048,  0.5373964 ]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.1334256 , -0.25891632],
        [ 0.12293333,  0.20609343]], dtype=float32)>)

In [ ]:
# x: [B, N1, N2, N3, N4, N5, N6, C]
    B, N1, N2, C = x.shape

    # Step 1: Convolve over (N1, N2, N3)
    x1 = tf.reshape(x, [B * N2, N1, C])
    y1 = tf.nn.convolution(x1, kernel1d, padding='SAME')
    y1 = tf.reshape(y1, [B, N1, N2, C])
   
    # Step 2: Convolve over (N4, N5, N6)
    x2 = tf.reshape(y1, [B * N1, N2, C])
    y2 = tf.nn.convolution(x2, kernel3d, padding='SAME')
    y2 = tf.reshape(y2, [B, N1, N2, C])

In [294]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

# Separable 3D convolution try
        
        separable and nonseparable (referencd) 3d convolution
        outer product of kernel
        Perfect match with one channel input
        !! does not match when multiple channel input

In [ ]:
# import tensorflow as tf

# def separable_3d_conv(x, kernel1d):
#     # x: [B, D1, D2, D3, C]
#     B, D1, D2, D3, C = x.shape

#     # Step 1: Convolve over D1
#     x1 = tf.transpose(x, [0, 2, 3, 1, 4])           # [B, D2, D3, D1, C]
#     x1 = tf.reshape(x1, [B * D2 * D3, D1, C])       # [B*D2*D3, D1, C]
#     y1 = tf.nn.convolution(x1, kernel1d, padding='SAME')
#     y1 = tf.reshape(y1, [B, D2, D3, D1, C])
#     y1 = tf.transpose(y1, [0, 3, 1, 2, 4])          # [B, D1, D2, D3, C]

#     # Step 2: Convolve over D2
#     x2 = tf.transpose(y1, [0, 1, 3, 2, 4])          # [B, D1, D3, D2, C]
#     x2 = tf.reshape(x2, [B * D1 * D3, D2, C])
#     y2 = tf.nn.convolution(x2, kernel1d, padding='SAME')
#     y2 = tf.reshape(y2, [B, D1, D3, D2, C])
#     y2 = tf.transpose(y2, [0, 1, 3, 2, 4])          # [B, D1, D2, D3, C]

#     # Step 3: Convolve over D3
#     x3 = tf.reshape(y2, [B * D1 * D2, D3, C])
#     y3 = tf.nn.convolution(x3, kernel1d, padding='SAME')
#     y3 = tf.reshape(y3, [B, D1, D2, D3, C])

#     return y3


In [360]:
import tensorflow as tf

def separable_3d_conv(x, kernel1d):
    # x: [B, N1, N2, N3, N4, N5, N6, C]
    B, N1, N2, N3, C = x.shape

    # Step 1: Convolve over (N1)
    ## B, N1, N2, N3, C
    x1 = tf.reshape(x, [B * N2 * N3, N1, C])
    y1 = tf.nn.convolution(x1, kernel1d, padding='SAME')
    y1 = tf.reshape(y1, [B, N1, N2, N3, C])
    y1 = tf.transpose(y1, perm=[0,2,1,3,4]) 
    ## B, N2, N1, N3, C
    
    # Step 2: Convolve over (N2)
    ## B, N1, N2, N3, C
    x2 = tf.reshape(y1, [B * N1 * N3, N2, C])
    y2 = tf.nn.convolution(x2, kernel1d, padding='SAME')
    y2 = tf.reshape(y2, [B, N2, N1, N3, C])
    y2 = tf.transpose(y2, perm=[0,2,1,3,4]) ## B, N1, N2, N3, C
    y2 = tf.transpose(y2, perm=[0,3,2,1,4]) ## B, N3, N2, N1, C
    
    
    # Step 3: Convolve over (N3)
    ## B, N2, N1, N3, C
    x3 = tf.reshape(y2, [B * N1 * N2, N3, C])
    y3 = tf.nn.convolution(x3, kernel1d, padding='SAME')
    y3 = tf.reshape(y3, [B, N3, N1, N2, C])
    y3 = tf.transpose(y3, perm=[0,3,2,1,4]) 
    ## B, N1, N2, N3, C

    return y3

# -------------------------------
# Test the function with dummy data
# -------------------------------

# Input shape
B, N1, N2, N3, C = 1, 8, 8, 8, 1 #OK
B, N1, N2, N3, C = 1, 2, 2, 2, 1 #OK
# B, N1, N2, C = 1, 2, 2, 2 # 
# B, N1, N2, C = 50, 8, 8, 1 #OK
# B, N1, N2, C = 2, 2, 2, 2 #issue with multiple channels
# B, N1, N2, C = 5, 2, 3, 2
# B, N1, N2, C = 5, 8, 8, 2

# O = 2
x = tf.random.normal((B, N1, N2, N3, C))

# Shared 2D convolution kernel (C -> C to keep channels same)
kH = 3
kH = 2
kernel1d = tf.random.normal((kH, C, C))  # single 2D kernel reused

# Run the separable 4D convolution
y = separable_3d_conv(x, kernel1d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel1d.shape)
print("Output shape:", y.shape)



Input shape : (1, 2, 2, 2, 1)
Kernel 2D shape: (2, 1, 1)
Output shape: (1, 2, 2, 2, 1)


In [361]:
kernel3d = tf.einsum('ico,jco,kco->ijkco', kernel1d, kernel1d, kernel1d)
kernel3d.shape

TensorShape([2, 2, 2, 1, 1])

In [362]:
ydef = tf.nn.convolution(x, kernel3d, padding='SAME')
ydef.shape

TensorShape([1, 2, 2, 2, 1])

In [363]:
# Compare
diff = tf.reduce_max(tf.abs(y - ydef))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(y - ydef) < 1e-3).numpy())


Max absolute difference: 1.9303341
Outputs match: False


In [304]:
B, N1, N2, N3, C = 1, 8, 9, 10, 1 #OK
x = tf.random.normal((B, N1, N2, N3, C))

In [305]:
# B, N1, N2, N3, N4, C = x.shape

_ = tf.reshape(x, [B * N2 * N3, N1, C])  # shape: (B*N3*N4, N1, N2, C)
# print(f'x {x.shape} \nx1 ', x1.shape)
# y1 = tf.nn.convolution(x1, kernel2d, padding='SAME')
# print('y1 ', y1.shape)
_r = tf.reshape(_, [B, N1, N2, N3, C]) 
x.shape, _r.shape

(TensorShape([1, 8, 9, 10, 1]), TensorShape([1, 8, 9, 10, 1]))

In [315]:

# # Compare
diff = tf.reduce_max(tf.abs(x - _r))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(x - _r) < 1e-4).numpy())

Max absolute difference: 0.0
Outputs match: True


In [314]:
_ = tf.transpose(x, perm=[0,2,1,3,4])
_ = tf.reshape(_, [B * N1 * N3, N2, C])
# y2 = tf.nn.convolution(x2, kernel1d, padding='SAME')
    # y2 = tf.reshape(y2, [B, N1, N2, C])
    # y2 = tf.transpose(y2, perm=[0,2,1,3])
_r = tf.reshape(_, [B, N2, N1, N3, C])
_r = tf.transpose(_r, perm=[0,2,1,3,4])

In [ ]:
# def foo(x):
#     C = 1
#     tf.expand_dims(tf.unstack(x, axis=-1))

[<tf.Tensor: shape=(1, 2, 2), dtype=float32, numpy=
 array([[[ 0.96406895,  1.7949798 ],
         [-0.18434155, -0.16262998]]], dtype=float32)>,
 <tf.Tensor: shape=(1, 2, 2), dtype=float32, numpy=
 array([[[ 0.25728133, -0.1631882 ],
         [ 0.96027887,  0.08398907]]], dtype=float32)>]

In [ ]:
kernel2d = tf.einsum('ico,kco->ikco', kernel1d, kernel1d)
kernel2d.shape

TensorShape([2, 2, 2, 2])

In [ ]:
ydef = tf.nn.convolution(x, kernel2d, padding='SAME')
ydef.shape

TensorShape([1, 2, 2, 2])

In [ ]:
# Compare
diff = tf.reduce_max(tf.abs(y - ydef))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(y - ydef) < 1e-4).numpy())


Max absolute difference: 0.7652533
Outputs match: False


# Limitations of existing tf libraries

The following two blocks shows limitations of tf.nn.convolution beyond 3D conv

In [ ]:
import tensorflow as tf

# Input: (batch, N1, N2, N3, N4, in_channels)
x = tf.random.normal((2, 16, 16, 16, 16, 3))  # Batch=2, 4D spatial dims, 3 channels

# Kernel: (K1, K2, K3, K4, in_channels, out_channels)
kernel = tf.random.normal((3, 3, 3, 3, 3, 8))  # 4D kernel, 3 in_ch, 8 out_ch

# Strides and padding
strides = [1, 1, 1, 1, 1]  # [batch, N1, N2, N3, N4] (no stride on batch/channels)
padding = "SAME"  # or "VALID"

# Perform ND convolution (supports 4D!)
output = tf.nn.convolution(
    input=x,          # (batch, N1, N2, N3, N4, in_channels)
    filters=kernel,   # (K1, K2, K3, K4, in_channels, out_channels)
    strides=strides[1:-1],  # Strides for spatial dims only [N1, N2, N3, N4]
    padding=padding,
    data_format="NDHWC"  # Explicitly specify 4D spatial dims
)

print(output.shape)  # Output: (2, 16, 16, 16, 16, 8)

ValueError: `num_spatial_dims` must be 1, 2, or 3. Received: num_spatial_dims=4.

In [ ]:
import tensorflow as tf

# Define input tensor: [batch, D1, D2, D3, channels]
input_tensor = tf.random.normal([1, 10, 10, 10, 10, 3])

# Define filter/kernel tensor: [K1, K2, K3, in_channels, out_channels]
filter_tensor = tf.random.normal([3, 3, 3, 3, 3, 8])

# Perform convolution
output = tf.nn.convolution(
    input=input_tensor,
    filters=filter_tensor,
    padding='SAME',
    strides=[1, 1, 1],
    dilations=[1, 1, 1]
)

print("Output shape:", output.shape)


ValueError: `num_spatial_dims` must be 1, 2, or 3. Received: num_spatial_dims=4.

In [ ]:
# Input: (batch, D1, D2, D3, D4, channels)
input_tensor = tf.random.normal((2, 8, 8, 8, 8, 3))

# Filter: (3, 3, 3, 3, in_channels, out_channels)
filters = tf.random.normal((3, 3, 3, 3, 3, 5))

# Convolution with 4D spatial + channels-last format
output = tf.nn.convolution(
    input=input_tensor,
    filters=filters,
    strides=[1, 1, 1, 1],
    padding='SAME',
    # data_format='NDHWC'  # ✅ correct
)


ValueError: `num_spatial_dims` must be 1, 2, or 3. Received: num_spatial_dims=4.

The follwing implementation is incorrect

In [ ]:
import tensorflow as tf

#%% ConvND layer ""This in incorrect! due to filter size
class ConvND(tf.keras.layers.Layer):
    """ND input --kkt@30-08-2024"""
    def __init__(self, 
        filters=1, 
        kernel_size=3, **kwargs):
        super(ConvND, self).__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Initialize the bias (a0) for the numerator polynomial
        self.h0 = self.add_weight(shape=(), initializer='zeros', trainable=True)#, name='h0_bias')
        
        # ND kernel
        # Create the convolution kernel
        self.kernelND = self.add_weight(
            shape=(self.kernel_size, input_shape[-1], self.filters),
            initializer='glorot_uniform',
            trainable=True
        )

    def call(self, x):
        y = tf.nn.convolution(
            x,
            filters=self.kernelND, 
            padding='SAME')
        return y

    def get_config(self):
        config = super(ConvND, self).get_config()
        return config